In [11]:
import os
import json
from IPython.display import Markdown
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType,
)

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0 pyspark-shell"
)

if os.name == "nt" and "HADOOP_HOME" in os.environ:
    os.environ["PATH"] = (
        os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]
    )


with open("API.json", encoding="utf-8") as f:
    kafka_config = json.load(f)

jaas_config = (
    "org.apache.kafka.common.security.plain.PlainLoginModule required "
    f'username="{kafka_config["username"]}" password="{kafka_config["password"]}";'
)

my_name = "serhii"
sensors_topic = f"{my_name}_iot_sensors"
alerts_topic = f"{my_name}_iot_alerts"

## Крок 1. Генерація даних сенсорів

In [12]:
admin_client = KafkaAdminClient(
    bootstrap_servers=kafka_config["bootstrap_servers"],
    security_protocol=kafka_config["security_protocol"],
    sasl_mechanism=kafka_config["sasl_mechanism"],
    sasl_plain_username=kafka_config["username"],
    sasl_plain_password=kafka_config["password"],
)

num_partitions = 2
replication_factor = 1

topic_names = [sensors_topic, alerts_topic]

existing_topics = admin_client.list_topics()
new_topics = [
    NewTopic(
        name=name, num_partitions=num_partitions, replication_factor=replication_factor
    )
    for name in topic_names
    if name not in existing_topics
]

try:
    if new_topics:
        admin_client.create_topics(new_topics=new_topics, validate_only=False)
        for topic in new_topics:
            print(f"Topic '{topic.name}' created successfully.")
    else:
        print("All topics already exist.")
except TopicAlreadyExistsError:
    print("Topics already exist.")
except Exception as e:
    print(f"An error occurred: {e}")

[
    print(topic)
    for topic in admin_client.list_topics()
    if topic.startswith(f"{my_name}_iot")
]

admin_client.close()

All topics already exist.
serhii_iottest_sensors
serhii_iottest2_alerts
serhii_iottest2_sensors
serhii_iottest_alerts
serhii_iot_sensors
serhii_iot_alerts


Генератор винесено в `sensor.py`. Змінено поля (`id, temperature, humidity, timestamp`), топік і спосіб генерації: значення змінюються плавно (випадкове блукання), бо середнє незалежного шуму за хвилину завжди близьке до центру діапазону й ніколи не дає алертів. Один запуск — один датчик зі своїм `id`:120 повідомлень з інтервалом 2 с, тобто ~4 хв даних. Запуск в окремому терміналі (можна в кількох одночасно):

```powershell
python sensor.py
```

Вміст `sensor.py`:

In [13]:
with open("sensor.py", encoding="utf-8") as f:
    sensor_code = f.read()

Markdown(f"```python\n{sensor_code}\n```")

```python
from kafka import KafkaProducer
from kafka.serializer import Serializer
from pathlib import Path
import json
import uuid
import time
import random


class JsonSerializer(Serializer):
    def serialize(self, topic, headers, data):
        if data is None:
            return None
        return json.dumps(data).encode("utf-8")


config_path = Path(__file__).parent / "API.json"
with open(config_path, encoding="utf-8") as f:
    kafka_config = json.load(f)

producer = KafkaProducer(
    bootstrap_servers=kafka_config["bootstrap_servers"],
    security_protocol=kafka_config["security_protocol"],
    sasl_mechanism=kafka_config["sasl_mechanism"],
    sasl_plain_username=kafka_config["username"],
    sasl_plain_password=kafka_config["password"],
    value_serializer=JsonSerializer(),
    key_serializer=JsonSerializer(),
)


my_name = "serhii"
sensors_topic = f"{my_name}_iot_sensors"


sensor_id = random.randint(1000, 9999)
num_messages = 120

temperature = random.uniform(25, 45)
humidity = random.uniform(15, 85)

print(f"Sensor {sensor_id} started")

try:
    for i in range(num_messages):
        temperature = min(max(temperature + random.uniform(-2, 2), 10), 60)
        humidity = min(max(humidity + random.uniform(-4, 4), 0), 100)
        data = {
            "id": sensor_id,
            "temperature": round(temperature, 2),
            "humidity": round(humidity, 2),
            "timestamp": time.time(),
        }
        producer.send(sensors_topic, key=str(uuid.uuid4()), value=data)
        producer.flush()
        print(f"Message {i} sent to topic '{sensors_topic}' successfully. Data: {data}")
        time.sleep(2)
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    producer.close()

```

### Вивід консолі


![Крок 1](screenshots/P1_1.png)
![Крок 1](screenshots/P1_2.png)

## Крок 2. Знаходження середніх значень показників

Читаємо потік із Kafka, розбираємо JSON і рахуємо середні температуру й вологість у sliding window: довжина 1 хвилина, крок 30 секунд, watermark 10 секунд.

In [ ]:
spark = (
    SparkSession.builder.appName("IoTAlertsStreaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_config["bootstrap_servers"][0])
    .option("kafka.security.protocol", kafka_config["security_protocol"])
    .option("kafka.sasl.mechanism", kafka_config["sasl_mechanism"])
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("subscribe", sensors_topic)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "50")
    .load()
)

json_schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("temperature", DoubleType(), True),
        StructField("humidity", DoubleType(), True),
        StructField("timestamp", DoubleType(), True),
    ]
)

sensors_df = (
    df.selectExpr("CAST(value AS STRING) AS value_deserialized")
    .withColumn("value_json", from_json(col("value_deserialized"), json_schema))
    .select(
        col("value_json.id").alias("id"),
        col("value_json.temperature").alias("temperature"),
        col("value_json.humidity").alias("humidity"),
        timestamp_seconds(col("value_json.timestamp")).alias("timestamp"),
    )
)

avg_df = (
    sensors_df.withWatermark("timestamp", "10 seconds")
    .groupBy(window(col("timestamp"), "1 minute", "30 seconds"))
    .agg(
        avg("temperature").alias("t_avg"),
        avg("humidity").alias("h_avg"),
    )
)

## Крок 3. Побудова фільтраційного механізму

Параметри алертів читаються з `alerts_conditions.csv`. Значення `-999` означає, що межа не використовується.

In [ ]:
conditions_schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("humidity_min", DoubleType(), True),
        StructField("humidity_max", DoubleType(), True),
        StructField("temperature_min", DoubleType(), True),
        StructField("temperature_max", DoubleType(), True),
        StructField("code", StringType(), True),
        StructField("message", StringType(), True),
    ]
)

conditions_df = (
    spark.read.option("header", True)
    .schema(conditions_schema)
    .csv("alerts_conditions.csv")
)

conditions_df.show(truncate=False)

+---+------------+------------+---------------+---------------+----+-------------+
|id |humidity_min|humidity_max|temperature_min|temperature_max|code|message      |
+---+------------+------------+---------------+---------------+----+-------------+
|1  |0.0         |40.0        |-999.0         |-999.0         |101 |It's too dry |
|2  |60.0        |100.0       |-999.0         |-999.0         |102 |It's too wet |
|3  |-999.0      |-999.0      |-300.0         |30.0           |103 |It's too cold|
|4  |-999.0      |-999.0      |40.0           |300.0          |104 |It's too hot |
+---+------------+------------+---------------+---------------+----+-------------+



Cross join кожного вікна з усіма умовами, далі фільтр: середнє має потрапити в діапазон `(min, max)` умови, а межа `-999` вважається виконаною.

In [16]:
alerts_df = (
    avg_df.crossJoin(conditions_df)
    .filter(
        ((col("humidity_min") == -999) | (col("h_avg") > col("humidity_min")))
        & ((col("humidity_max") == -999) | (col("h_avg") < col("humidity_max")))
        & ((col("temperature_min") == -999) | (col("t_avg") > col("temperature_min")))
        & ((col("temperature_max") == -999) | (col("t_avg") < col("temperature_max")))
    )
    .select(
        col("window"),
        col("t_avg"),
        col("h_avg"),
        col("code"),
        col("message"),
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss.SSSSSS").alias(
            "timestamp"
        ),
    )
)

## Крок 4. Запис даних у Kafka-топік

Кожен алерт серіалізовано в JSON і записано у `serhii_iot_alerts`. Режим `append`: вікно потрапляє у вихід один раз, коли watermark проходить його кінець, тому кожен алерт остаточний. Звідси й наслідок: вікно закривається лише тоді, коли надходять новіші дані, тож останні 1–2 вікна після зупинки сенсора у вихід не потрапляють.

In [ ]:
prepare_to_kafka_df = alerts_df.select(
    col("code").alias("key"),
    to_json(
        struct(
            col("window"),
            col("t_avg"),
            col("h_avg"),
            col("code"),
            col("message"),
            col("timestamp"),
        )
    ).alias("value"),
)

query = (
    prepare_to_kafka_df.writeStream.trigger(processingTime="10 seconds")
    .outputMode("append")
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_config["bootstrap_servers"][0])
    .option("kafka.security.protocol", kafka_config["security_protocol"])
    .option("kafka.sasl.mechanism", kafka_config["sasl_mechanism"])
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("topic", alerts_topic)
    .option("checkpointLocation", "checkpoints/alerts")
    .start()
)

query.awaitTermination(300)
query.stop()
print(query.lastProgress)

{
    "id": "ad34cbcd-e46f-4848-805f-061e6c31be9f",
    "runId": "90c9d428-7bf6-4fc2-8a73-b187c5da9637",
    "name": null,
    "timestamp": "2026-11-22T18:02:00.001Z",
    "batchId": 7,
    "batchDuration": 227,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 226,
        "triggerExecution": 227
    },
    "eventTime": {
        "watermark": "2026-11-22T17:48:36.439Z"
    },
    "stateOperators": [
        {
            "operatorName": "stateStoreSave",
            "numRowsTotal": 2,
            "numRowsUpdated": 0,
            "allUpdatesTimeMs": 0,
            "numRowsRemoved": 0,
            "allRemovalsTimeMs": 0,
            "commitTimeMs": 0,
            "memoryUsedBytes": 2112,
            "numRowsDroppedByWatermark": 0,
            "numShufflePartitions": 2,
            "numStateStoreInstances": 2,
            "customMetrics": {
                "SnapshotLastUploaded.partition_0_default": -1,
  

## Перевірка: вміст топіку алертів

In [18]:
alerts_check_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", kafka_config["bootstrap_servers"][0])
    .option("kafka.security.protocol", kafka_config["security_protocol"])
    .option("kafka.sasl.mechanism", kafka_config["sasl_mechanism"])
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("subscribe", alerts_topic)
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value")
)

print(f"Alerts in topic '{alerts_topic}': {alerts_check_df.count()}")
alerts_check_df.show(50, truncate=False)

Alerts in topic 'serhii_iot_alerts': 12
+---+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key|value                                                                                                                                                                                                                           |
+---+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|101|{"window":{"start":"2026-11-22T18:44:00.000+01:00","end":"2026-11-22T18:45:00.000+01:00"},"t_avg":30.110476190476188,"h_avg":33.67619047619047,"code":"101","message":"It's too dry","timestamp":"2026-11-22 18:57:22.651000"}  |
|101|{"window":{"start":"2026-11-22T

In [ ]:
first_alert = alerts_check_df.first()
if first_alert:
    print(json.dumps(json.loads(first_alert["value"]), indent=2, ensure_ascii=False))
else:
    print("No alerts yet.")

{
  "window": {
    "start": "2026-11-22T18:44:00.000+01:00",
    "end": "2026-11-22T18:45:00.000+01:00"
  },
  "t_avg": 30.110476190476188,
  "h_avg": 33.67619047619047,
  "code": "101",
  "message": "It's too dry",
  "timestamp": "2026-11-22 18:57:22.651000"
}


In [20]:
spark.stop()